# Data Cleaning

## Imports

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import zipfile


raw_data_path = "../data/raw_data/"
clean_data_path = "../data/cleaned_data/"

def save_clean_data(df, file_name):
    os.makedirs(clean_data_path, exist_ok=True)
    df.to_csv(clean_data_path + file_name, index=False)

def unzip_raw_data(zip_path="../data/data.zip", extract_to="../data/"):
    # Ensure output directory exists
    os.makedirs(extract_to, exist_ok=True)

    # Unzip
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)

    print(f"Uncompressed {zip_path} → {extract_to}")

unzip_raw_data()

### Wine data

In [ ]:
wine_red_data_raw = pd.read_csv(raw_data_path + "winequality/winequality-red.csv", sep=";")
wine_white_data_raw = pd.read_csv(raw_data_path + "winequality/winequality-white.csv", sep=";")

### Retail data

In [ ]:
online_retail_data_raw = pd.read_excel(raw_data_path + "retail/Online Retail.xlsx")

### Cup98

In [ ]:
cup98_data_raw = pd.read_csv(raw_data_path + "cup98/cup98LRN.txt")

In [ ]:
def visualize_missing(df_v, name, sample_n=1000):
    ncols = df_v.shape[1]
    miss_pct = df_v.isna().mean().sort_values(ascending=False) * 100

    # MUCH larger figure and more spacing
    fig = plt.figure(constrained_layout=True, figsize=(18, 12))
    gs = fig.add_gridspec(
        2, 3,
        height_ratios=[1.2, 1.5],   # make heatmap row bigger
        width_ratios=[2, 2, 1.2]     # give barplot more room
    )

    # --- BARPLOT (Top Missing Columns) ---
    ax0 = fig.add_subplot(gs[0, :2])
    sns.barplot(
        x=miss_pct.values[:40],
        y=miss_pct.index[:40],
        palette="viridis",
        ax=ax0
    )
    ax0.set_xlabel("Missing %", fontsize=12)
    ax0.set_title(f"{name} — Top Missing Columns (up to 40)", fontsize=14)
    ax0.tick_params(labelsize=10)
    ax0.grid(axis="x", linestyle="--", alpha=0.3)

    # --- INFO PANEL ---
    ax1 = fig.add_subplot(gs[0, 2])
    ax1.axis("off")
    ax1.text(
        0, 0.7,
        f"Rows: {len(df_v)}\n"
        f"Columns: {ncols}\n"
        f"Columns w/ missing: {(miss_pct>0).sum()}\n"
        f"Max missing %: {miss_pct.max():.1f}%",
        fontsize=12
    )

    # --- HEATMAP ---
    sample = df_v if len(df_v) <= sample_n else df_v.sample(sample_n, random_state=0)
    miss_mat = sample.isna().astype(int).T

    max_cols_for_heatmap = 80
    if miss_mat.shape[1] > max_cols_for_heatmap:
        miss_mat = miss_mat.iloc[:, :max_cols_for_heatmap]
        title = f"{name} — Missingness Heatmap\n(sample {len(sample)} rows, first {max_cols_for_heatmap} columns)"
    else:
        title = f"{name} — Missingness Heatmap\n(sample {len(sample)} rows)"

    ax2 = fig.add_subplot(gs[1, :])

    # Increase heatmap cell size
    sns.heatmap(
        miss_mat,
        cbar=False,
        cmap=["#2b8cbe", "#f7f7f7"],
        ax=ax2,
        linewidths=0,
    )

    ax2.set_title(title, fontsize=14)
    ax2.set_ylabel("Columns (transposed)", fontsize=12)
    ax2.set_xlabel("Sampled Rows", fontsize=12)
    ax2.tick_params(labelsize=9)

    plt.show()


In [ ]:
visualize_missing(wine_red_data_raw, "wine_red_data_raw", sample_n=500)
visualize_missing(wine_white_data_raw, "wine_white_data_raw", sample_n=500)
visualize_missing(online_retail_data_raw, "online_retail_data_raw", sample_n=2000)
visualize_missing(cup98_data_raw, "cup98_data_raw", sample_n=2000)

## Online_Retail

If we are analyzing individual customer behavior, there is no need for ones without ID. As they are probably one-time purchasers that add not significant effect to our modeling goal

In [ ]:
online_retail_data_raw.head()

In [ ]:
online_retail_data = online_retail_data_raw.copy()
online_retail_data = online_retail_data.dropna(subset=["CustomerID"])

In [ ]:
save_clean_data(online_retail_data, "online_retail_data_clean.csv")

## winequality

In [ ]:
wine_red_data_raw.head()

In [ ]:
wine_white_data_raw.head()

In [ ]:
save_clean_data(wine_red_data_raw, "wine_red_data_clean.csv")
save_clean_data(wine_white_data_raw, "wine_white_data_clean.csv")

## Cup98

In this study: https://lowrank.net/nikos/pubs/empirical.pdf
They did mean imputation for missing values, and everything thats missing was literally just "MISSING"
- Here I dropped all columns where >50% of the values are missing

“KDD98 is from the 1998 KDD-Cup. The task is to predict if a person donates money.
This is the only dataset with missing values. We impute the mean for continuous features and treat missing nominal and boolean features as new values.” (Page 3)

In [ ]:
def impute_cup98(df):
    df = df.copy()

    for col in df.columns:

        # --- 1. Try converting to numeric to detect mixed-type numeric/binary flags
        coerced = pd.to_numeric(df[col], errors="coerce")

        # Case A: Column *becomes* numeric after coercion → treat as numeric (binary or continuous)
        if coerced.notna().sum() > 0 and coerced.isna().sum() < len(df):
            # Use coerced version when possible
            df[col] = coerced
            # Numeric imputation → mean
            df[col] = df[col].fillna(df[col].mean())
            continue

        # Case B: Column does NOT coerce to numeric → treat as categorical
        else:
            # Replace missing-like tokens with "MISSING"
            df[col] = df[col].astype(str).replace(["", " ", "X", "nan", "NaN"], "MISSING")
            df[col] = df[col].fillna("MISSING")
            continue

    return df

In [ ]:
cup98_data_raw = cup98_data_raw.loc[:, cup98_data_raw.isna().mean() < 0.50]
cup98_data_clean = impute_cup98(cup98_data_raw)


save_clean_data(cup98_data_clean, "cup98_data_clean.csv")